In [1]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
import os
%matplotlib widget
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size':8})
pd.options.display.max_columns = 200
pd.options.display.max_rows = 300
pd.options.display.max_seq_items = 2000
# from tqdm import tqdm
from datetime import timedelta
from datetime import datetime, time
import re
import sys
sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import load_sleep_data, write_to_hdf5_file, get_metadata, read_in_routine
from tqdm import tqdm

from medications import *

### define project name and required/optional paths to EDW output

To keep this notebook clean, have preprocessing for data needed done in another notebook - those are usually project specific.

Files needed:  
1.) 'cohort' file with columns:  
    a) 'PatientID' (zPatientID)  
    b) AdmissionDTS  
    c) DischargeDTS  
    d) Weight (optional)
      
2.) EDW pull outputs:  
    a) medications  
    b) flowsheets for weight  
    c) ADT data  
    
3.) Medication Dictionary (meds to study) table with columns:  
    a) med_name  
    b) erx (optional)  
    
Reason for not justing processing all medications is that the EDW output fromat of medications is quite irregular. For many medications, I had to manually check how to fix it. So it's better to only process medications that are needed and not run into unnecessary issues.  
2 ways of defining medications of interest
a) list of medications or  
b) a list of medications together with their ERX numbers.  
ERX numbers are more specific. E.g. providing only medication name might lead to unwanted results as some medications are given via eyedrop in very low dose etc.

In [2]:
# SET PATHS FOR THE FILES NEEDED:

# project = 'covid_sedation'
project = 'covid_sedation'

if project == 'covid_sedation':
    
    surge_add = ''
#     surge_add = '2'
    
    cohort_path = f'/media/mad3/Projects/Wolfgang/Covid19_Sedation/cohort_covid_sedation{surge_add}.csv'
    edw_meds_path = f'/media/mad3/Projects/Wolfgang/Covid19_Sedation/edw_covid_sedation{surge_add}_meds.csv'
    edw_adt_path = f'/media/mad3/Projects/Wolfgang/Covid19_Sedation/edw_covid_sedation{surge_add}_adt.csv'
    edw_flowsheets_path = f'/media/mad3/Projects/Wolfgang/Covid19_Sedation/edw_covid_sedation{surge_add}_flowsheets.csv'
    meds_dictionary_path = '/media/mad3/Projects/Wolfgang/Covid19_Sedation/medication_dictionary.csv'
    equivalency_version = 'covid_sedation'
    equivalency_table_path = f'/scr/wolfgang/Covid19_Sedation/equivalency_table_{equivalency_version}.csv'
    # output:
    meds_output_dir = f'/scr/wolfgang/Covid19_Sedation/medications_output{surge_add}'
    meds_output_dir_minute = f'/scr/wolfgang/Covid19_Sedation/medications_all_timeseries_minute{surge_add}'
    meds_output_dir_minute_wroute = f'/scr/wolfgang/Covid19_Sedation/medications_all_timeseries_minute_wroute{surge_add}'
    meds_output_dir_hour = f'/scr/wolfgang/Covid19_Sedation/medications_all_timeseries_hour{surge_add}'
    meds_output_dir_day = f'/scr/wolfgang/Covid19_Sedation/medications_all_timeseries_day{surge_add}'

elif project == 'covid_delirium':
    
    cohort_path = '/scr/wolfgang/Covid19_Delirium/cohort.csv'
    edw_meds_path = '/scr/wolfgang/Covid19_Delirium/edw_covid_delirium_meds.csv'
    edw_adt_path = '/scr/wolfgang/Covid19_Delirium/edw_covid_delirium_adt.csv'
    edw_flowsheets_path = '/scr/wolfgang/Covid19_Delirium/edw_covid_delirium_flowsheets.csv'
    meds_dictionary_path = '/scr/wolfgang/Covid19_Delirium/medication_dictionary.csv'
    equivalency_version = 'covid_delirium'
    equivalency_table_path = f'/scr/wolfgang/Covid19_Delirium/equivalency_table_{equivalency_version}.csv'
    # directory where the subject-wise timeseries shall be saved:
    meds_output_dir = '/scr/wolfgang/Covid19_Delirium'
    meds_output_dir_minute = '/scr/wolfgang/Covid19_Delirium/medications_all_timeseries_minute'
    meds_output_dir_minute_wroute = '/scr/wolfgang/Covid19_Sedation/medications_all_timeseries_minute_wroute'
    meds_output_dir_hour = '/scr/wolfgang/Covid19_Delirium/medications_all_timeseries_hour'
    meds_output_dir_day = '/scr/wolfgang/Covid19_Delirium/medications_all_timeseries_day'

for dir_tmp in [meds_output_dir, meds_output_dir_minute, meds_output_dir_hour,
                meds_output_dir_day, meds_output_dir_minute_wroute]:
    if not os.path.exists(dir_tmp):
        os.makedirs(dir_tmp)


### load data

In [3]:
cohort = pd.read_csv(cohort_path)
flowsheets_data = pd.read_csv(edw_flowsheets_path)
adt_data = pd.read_csv(edw_adt_path)
meds_data = pd.read_csv(edw_meds_path)
meds_dictionary = pd.read_csv(meds_dictionary_path)

if project == 'covid_sedation':
    if surge_add == '':
        cohort.loc[cohort.PatientID == 'Z10292841', 'Weight'] = 94.5
        cohort.loc[cohort.PatientID == 'Z17768076', 'Weight'] = 46.7
    if surge_add == '2':
        cohort.rename({'Admit Date':'AdmissionDTS', 'Discharge Date': 'DischargeDTS'}, axis=1, inplace=True)

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/IPython/core/interactiveshell.py:3063: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)
/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/IPython/core/interactiveshell.py:3063: DtypeWarning: Columns (15,17,18,21,24,25,27,28,29,32,33,37) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [4]:
cohort.columns

Index(['PatientID', 'AdmissionDTS', 'DischargeDTS', 'First_ICU_Admit',
       'First_ICU_Discharge', 'Second_ICU_Admit', 'Second_ICU_Discharge',
       'Third_ICU_Admit', 'Third_ICU_Discharge', 'Fourth_ICU_Admit',
       'Fourth_ICU_Discharge', 'Number_ICU_Admissions', 'Age, years',
       'Hospital LOS, days', 'ICU LOS, days', 'OnsetDTS', 'Hospital',
       'PatientEncounterID', 'MRN', 'BirthDTS', 'SexDSC', 'EthnicGroupDSC',
       'PatientRaceDSC', 'Extubation', 'Intubation', 'CMO', 'DNR', 'DNI',
       'Age', 'Weight', 'Height', 'PMH Neurological Disorder',
       'If PMH neuro disorder, describe', 'Stroke ', 'Coma ',
       'Clinical or Electrographc Seizure/Status Epilepticus', 'Neuroimaging ',
       'ECMO', 'Dialysis/CRRT ', 'CMO.1', 'In Hospital Death (yes/no)',
       'Date of death', '1stETTplacementDTS', '1stETTremovalDTS',
       '2ndETTplacementDTS', '2ndETTremovalDTS', '3rdETTplacementDTS',
       '3rdETTremovalDTS', '4thETTplacementDTS', '4thETTremovalDTS',
       'Durat

In [5]:
if 0:
    print(meds_data.shape)
    meds_data = meds_data.iloc[::1000, :]
    print(meds_data.shape)

In [6]:
meds_data.columns

Index(['OrderID', 'PatientID', 'MRN', 'PatientEncounterID', 'MedicationID',
       'MedicationDSC', 'OrderInstantDTS', 'OrderStartDTS', 'OrderEndDTS',
       'DoseUnitCD', 'DoseUnitDSC', 'MinimumDoseAMT', 'MaximumDoseAMT',
       'PatientLocationID', 'PatientLocationDSC', 'MedicationTakenDTS',
       'MARActionCD', 'MARActionDSC', 'RouteDSC', 'RouteCD', 'SiteCD',
       'SiteDSC', 'InfusionRateNBR', 'InfusionRateUnitCD',
       'InfusionRateUnitDSC', 'DurationNBR', 'DurationUnitCD',
       'DurationUnitDSC', 'ActionSourceDSC', 'CommentTxt',
       'DefinedDailyDoseNBR', 'SigTXT', 'SigTXT.1', 'PrescriptionQuantityNBR',
       'MedicationDiscontinueReasonDSC', 'DiscreteFrequencyDSC',
       'DiscreteDoseAMT', 'DiscreteDispenseUnitDSC'],
      dtype='object')

In [7]:
assert(all(np.isin(['PatientID', 'AdmissionDTS', 'DischargeDTS'], cohort.columns))), 'Need those three columns!'
assert(all(np.isin(['PatientID'], meds_data.columns))), 'Need PatientID in meds_data!'

if 'erx' in meds_dictionary.columns:
    meds_dictionary_type = 'erx'
else:
    meds_dictionary_type = 'name'

In [8]:
print('Weight' in cohort.columns)
if 'Weight' not in cohort.columns:
    
    weight_data = flowsheets_data[(flowsheets_data.FlowsheetMeasureNM=='WEIGHT/SCALE')].copy()
    weight_data['weight_kg'] = (weight_data['MeasureTXT'].astype('float')/35.274).round(1)
    weight_data['RecordedDTS'] = pd.to_datetime(weight_data['RecordedDTS'], infer_datetime_format=1)

    cohort['Weight'] = np.nan

    for jloc, row in cohort.iterrows():

        zno_tmp = row['PatientID']

        weight_data_mrn = weight_data[weight_data['PatientID'] == zno_tmp]
        weight_data_mrn = weight_data_mrn[weight_data_mrn.RecordedDTS > row.AdmissionDTS]
        weight = np.nanmedian(weight_data_mrn.weight_kg.astype(float))
        cohort.loc[jloc, 'Weight'] = weight


True


In [9]:
def determine_IsInfusion(df):
    '''
    IsInfusion is True if DiscreteFrequencyDSC says 'Continuous' or if, for a whole order, DiscreteDoseAMT is empty but InfusionRateNBR is available.
    '''
    
    df['IsInfusion'] = (df.DiscreteFrequencyDSC=='Continuous PRN') | (df.DiscreteFrequencyDSC=='Continuous')
    
    for order_id in pd.unique(df.OrderID):
        df_order = df[df.OrderID == order_id]
        if (np.all(pd.isna(df_order.DiscreteDoseAMT)) & (~np.any(pd.isna(df_order.InfusionRateNBR)))):
            index_order = df_order.index
            df.loc[index_order, 'IsInfusion'] = True
    
    return df


def determine_dose_and_unit(df):
    '''
    Aim: Here, to the EDW output, three columns are appended: IsInfusion, Dose, Unit
    IsInfusion: boolean indicator if entry belongs to continuous infusion.
    Dose: Dose (in mg or mg/h)
    Unit: colum with 'mg' or 'mg/h'
    
    Input: dataframe, raw EDW medication output, after running preprocess_medication_data().
    Output: dataframe, raw EDW medication output + 3 additional columns, see above.
    
    This function therefore contains all the logic to determine and convert the original doses and units that are found in various fields, depending also on the type of medication.
    The output as additional columns will make further processing easy and human-read-friendly.
    '''
    
    df['IsInfusion'] = np.nan
    df['Unit'] = np.nan
    df['Dose'] = np.nan
    
    df = determine_IsInfusion(df)
    
    

In [10]:
valid_units = ['mg', 'mg/hr', 'g', 'g/hr', 'mcg', 'mcg/hr', 'mg/min', 'mcg/min', 'mcg/hr', ]
weight_dependent_units = ['mcg/kg/min', 'mcg/kg/hr', 'mcg/kg', 'mg/kg', 'g/kg']

def valid_unit_and_order_dose(df):
    
   # Check for good DoseUnitDSC and SigTXT_Order combination:
    valid_dose_order_and_unit = df.loc[(pd.isna(df.Unit)) & \
                                       (df.DoseUnitDSC.apply(lambda x: x in valid_units)) & \
                                       (pd.notna(df.SigTXT_Order)) & \
                                       (df.SigTXT_Order.apply(lambda x: type(x) in [int, float]))].index
    df.loc[valid_dose_order_and_unit, 'Unit'] = df.loc[valid_dose_order_and_unit, 'DoseUnitDSC'] 
    df.loc[valid_dose_order_and_unit, 'Dose'] = df.loc[valid_dose_order_and_unit, 'SigTXT_Order']  
    
    return df


def valid_unit_and_discrete_dose(df):
    
    # Check for good DoseUnitDSC and DiscreteDoseAMT combination:
    valid_discrete_dose_and_unit = df.loc[(pd.isna(df.Unit)) & \
                                           (df.DoseUnitDSC.apply(lambda x: x in valid_units)) & \
                                           (pd.notna(df.DiscreteDoseAMT)) & \
                                           (df.DiscreteDoseAMT.apply(lambda x: '-' not in str(x)))].index
    df.loc[valid_discrete_dose_and_unit, 'Unit'] = df.loc[valid_discrete_dose_and_unit, 'DoseUnitDSC']
    df.loc[valid_discrete_dose_and_unit, 'Dose'] = df.loc[valid_discrete_dose_and_unit, 'DiscreteDoseAMT'].astype(float)
    
    return df

def icu_sleep_studydrug(df):

    # Handle the ICU-Sleep Placebo/Study Drug, as it behave slightly different than other entries. We keep the original Infusion Rate and InfusionRate Unit, as we are blinded.
    icu_sleep_study_drug = df.loc[df.MedicationDSC.apply(lambda x: '2017P000090'.lower() in x.lower())].index
    df.loc[icu_sleep_study_drug, 'Unit'] = df.loc[icu_sleep_study_drug, 'InfusionRateUnitDSC']
    df.loc[icu_sleep_study_drug, 'Dose'] = df.loc[icu_sleep_study_drug, 'InfusionRateNBR']
    
    return df


def valid_infusion_rate_and_dose_from_med_name(df):
    
    # Entries with valid InfusionRate (mL/hr) and extraction of dose from Medicationname
    ml_h_infusion = df.loc[(pd.isna(df.Unit)) & \
                              (df.InfusionRateUnitDSC.apply(lambda x: str(x).lower() == 'ml/hr'))].index
    no_dose_found = []
    for jloc, row in df.loc[ml_h_infusion].iterrows():
        unit_dose = re.search(r"\d+ .{1,3}/ml?", row.MedicationDSC)
        unit_dose_mg_ml = re.search('\d+ mg/\d+ ml' , row.MedicationDSC)
        
        if unit_dose is not None:
            unit_dose = unit_dose[0]
            # make sure units match (e.g. /ml from MedicationDSC and ml/ from Infusionrate)
            dose = np.float(unit_dose.split(' ')[0]) * np.float(row.InfusionRateNBR)
            unit = unit_dose.split(' ')[1]
            assert unit.split('/')[1].lower() == row.InfusionRateUnitDSC.split('/')[0].lower()
            assert '/ml' in unit
            unit = unit.split('/')[0] + '/' + row.InfusionRateUnitDSC.split('/')[1].lower()
        
        elif unit_dose_mg_ml is not None:
            mg_ml = unit_dose_mg_ml[0]
            mg = np.float(mg_ml.split('/')[0].replace(' mg', ''))
            ml = np.float(mg_ml.split('/')[1].replace(' ml', ''))
            dose = mg/ml * np.float(row.InfusionRateNBR)
            
            assert row.InfusionRateUnitDSC.split('/')[0].lower() == 'ml'
            
            unit = 'mg/' + row.InfusionRateUnitDSC.split('/')[1].lower()

        else:
            if not row.MedicationDSC in no_dose_found:
                no_dose_found.append(row.MedicationDSC)
            continue

        df.loc[jloc, 'Dose'] = dose
        df.loc[jloc, 'Unit'] = unit
 
    if len(no_dose_found) > 0:
        print(f'Dose Parsing from MedicationDSC failed for {no_dose_found}.')
    
    return df


def infusions_sigtxt_and_med_name(df):
    
    # Entries with valid units in medication name and info about dose in SigTxt, verified for a few examples in Epic.
    candidates = df.loc[(pd.isna(df.Unit)) & \
                        (pd.isna(df.InfusionRateUnitDSC)) & \
                        (pd.isna(df.DiscreteDoseAMT)) & \
                        (df.MedicationDSC.apply(lambda x: 'ml' in str(x).lower() ))].index
    no_dose_found = []
    for jloc, row in df.loc[candidates].iterrows():
        unit_dose = re.search(r".{1,3}/ml?", row.MedicationDSC)
        unit_dose_mg_ml = re.search('\d+ mg/\d+ ml' , row.MedicationDSC)
        
        if unit_dose is not None:
            unit = unit_dose[0].replace('/ml', '').replace(' ', '')
            # make sure units match (e.g. /ml from MedicationDSC and ml/ from Infusionrate)
            dose = np.float(row.SigTXT_Order)

        elif unit_dose_mg_ml is not None:
            mg_ml = unit_dose_mg_ml[0]
            unit = 'mg'
            dose = np.float(row.SigTXT_Order)
            
        else:
            continue

        df.loc[jloc, 'Dose'] = dose
        df.loc[jloc, 'Unit'] = unit
 
    return df

def self_administered_meds(df, verbose=False):

    meds_self_controlled = [x for x in pd.unique(df.MedicationDSC) if ('PCA'.lower() in x) | ('PCEA'.lower() in x)]
    if verbose:
        print(f'For the following meds, patient-self-administration is assumed because of "PCA" or "PCEA" in Med Name. Placeholder dose of 0.01 is imputed.')
        print(meds_self_controlled)
    
    for med_self_controlled in meds_self_controlled:
        
        loc_self_controlled = df.loc[(pd.isna(df.Unit)) & \
                                    (df.MedicationDSC == med_self_controlled)].index
        loc_self_controlled_stopped = df.loc[(pd.isna(df.Unit)) & \
                                    (df.MedicationDSC == med_self_controlled) & \
                                    (df.MARActionDSC == 'Stopped')].index
        
                                    
        df.loc[loc_self_controlled, 'Unit'] = 'mg/hr'
        df.loc[loc_self_controlled, 'Dose'] = 0.01  
        df.loc[loc_self_controlled_stopped, 'Dose'] = 0
        
    return df


def patches(df):
    
    loc_patches_0 = df.loc[(pd.isna(df.Unit)) & \
                            (df.DoseUnitDSC=='patch') & \
                           (df.MARActionDSC.apply(lambda x: x in ['Patch Removed', 'Due']))].index
    df.loc[loc_patches_0, 'Dose'] = 0
    df.loc[loc_patches_0, 'Unit'] = 'mcg/hr'
    
    loc_patches_1 = df.loc[(pd.isna(df.Unit)) & \
                            (df.DoseUnitDSC=='patch') & \
                           (df.MARActionDSC.apply(lambda x: x in ['Patch Applied']))].index  
    
    for jloc, row in df.loc[loc_patches_1].iterrows():
        
        unit_dose_patch = re.search(r"(\d+[.])?\d+ .+? ", row.MedicationDSC)[0]
        dose = np.float(unit_dose_patch.split(' ')[0])
        unit = unit_dose_patch.split(' ')[1]
        if '/24' in unit: # that means descriptions says m(c)g/24 hours.
            dose = dose/24
            unit = unit.replace('24', 'hr')            

        df.loc[jloc, 'Dose'] = dose
        df.loc[jloc, 'Unit'] = unit

    return df


def tablets(df):
    '''Many tablets might be already handled by DiscreteDose+Unit processing script, however
    many entries seem to have no DiscreteDoseEntry, but Unit information is clear from Medication Name
    and dose can be extracted from SigTXT_Order'''
    
    loc_tablets = df.loc[(pd.isna(df.Unit)) & \
                           (df.MedicationDSC.apply(lambda x: np.any(['tablet' in x, 'capsule' in x]))) & \
                        (pd.notna(df.SigTXT_Order))].index
    
    for jloc, row in df.loc[loc_tablets].iterrows():
        
        unit = [x for x in [' mg ', ' g '] if x in row.MedicationDSC]
        assert len(unit) == 1, f'No or multiple Units found for tablet {row.MedicationDSC}.'
        
        unit = unit[0].replace(' ', '')
        try:
            dose = np.float(row.SigTXT_Order)
        except:
            continue
       
        df.loc[jloc, 'Dose'] = dose
        df.loc[jloc, 'Unit'] = unit
        
    # Similar, except Info is in DiscreteDose expect of SigTXT_ORder
    loc_tablets = df.loc[(pd.isna(df.Unit)) & \
                           (df.MedicationDSC.apply(lambda x: np.any(['tablet' in x, 'capsule' in x]))) & \
                        (pd.notna(df.DiscreteDoseAMT))].index
    
    for jloc, row in df.loc[loc_tablets].iterrows():
        
        unit = [x for x in [' mg ', ' g '] if x in row.MedicationDSC]
        assert len(unit) == 1, f'No or multiple Units found for tablet {row.MedicationDSC}.'
        
        unit = unit[0].replace(' ', '')
        try:
            dose = np.float(row.DiscreteDoseAMT)
        except:
            continue
            
        df.loc[jloc, 'Dose'] = dose
        df.loc[jloc, 'Unit'] = unit

    return df

def impute_missing_dose_unit(df, med_unit_dict):

    df['DoseUnitDSC_Imputed'] = 0
        
    for jloc, row in df.loc[pd.isna(df.Unit)].iterrows():

        unit = med_unit_dict[row.MedicationID]
        if len(unit) == 1:
            df.loc[jloc, 'DoseUnitDSC'] = unit[0]
            df.loc[jloc, 'DoseUnitDSC_Imputed'] = 1
            
    return df


def weight_dependent_dose(df):

    weight_dependent_dose = df.loc[(pd.isna(df.Unit)) & \
          (df.DoseUnitDSC.apply(lambda x: '/kg' in str(x)))].index

    for jloc, row in df.loc[weight_dependent_dose].iterrows():
        dose = None
        if pd.notna(row.SigTXT_Order):
            dose = np.float(row.SigTXT_Order)
        elif pd.notna(row.DiscreteDoseAMT):
            dose = np.float(row.DiscreteDoseAMT)
        if not dose is None:
            df.loc[jloc, 'Unit'] = row.DoseUnitDSC
            df.loc[jloc, 'Dose'] = dose
            
    return df


def use_mean_of_dose_range(df):

    dose_range_available = df.loc[pd.isna(df.Unit) & \
                (df.DiscreteDoseAMT.apply(lambda x: '-' in str(x))) & \
               (pd.isna(df.SigTXT_Order)) & \
               (pd.notna(df.DoseUnitDSC))].index

    df.loc[dose_range_available, 'Unit'] = df.loc[dose_range_available, 'DoseUnitDSC']
    df.loc[dose_range_available, 'Dose'] = df.loc[dose_range_available, 'DiscreteDoseAMT'].str.split('-').apply(lambda x: np.mean([float(y) for y in x]))

    return df


def sodium_chloride_iv(df):
    
    # 'sodium chloride 0.9 % intravenous solution'
    df.loc[(df.MedicationID == 27838), 'Dose'] = 0.09 * df.loc[(df.MedicationID == 27838), 'InfusionRateNBR']
    df.loc[(df.MedicationID == 27838), 'Unit'] = 'mg/hr'
    
    # 'sodium chloride 0.9 % bolus:
    df.loc[(df.MedicationID == 400291), 'Dose'] = 0.09 * df.loc[(df.MedicationID == 400291), 'SigTXT_Order']
    df.loc[(df.MedicationID == 400291), 'Unit'] = 'mg'
        
    # 'sodium chloride 0.45 % intravenous solution'
    df.loc[(df.MedicationID == 7318), 'Dose'] = 0.045 * df.loc[(df.MedicationID == 7318), 'InfusionRateNBR']
    df.loc[(df.MedicationID == 7318), 'Unit'] = 'mg/hr'
    
    # 'sodium chloride 0.45 % bolus:
    df.loc[(df.MedicationID == 400292), 'Dose'] = 0.045 * df.loc[(df.MedicationID == 400292), 'SigTXT_Order']
    df.loc[(df.MedicationID == 400292), 'Unit'] = 'mg'    
    
    return df

def set_isoflurane_to_dummy(df):
    '''Isoflurane is not well captured in the system. We can only make a binary statement, therefore fill values with dummy value 1.'''
    
    df.loc[(df.MedicationID == 127796), 'Dose'] = 1
    df.loc[(df.MedicationID == 127796), 'Unit'] = 'mg'
    
    return df


def init_medname_dict(df_mrn):
    print(df_mrn.columns)
    medname_original_new = {}
    for med in df_mrn['MedicationDSC'].unique():

        if '2017P000090'.lower() in med:
            new_name = 'DEXMEDETOMIDINE_OR_PLACEBO_(2017P000090)'.lower()
        else:
            new_name = med.replace(' ', '_').replace('-','_').replace('%', 'perc').replace('/', 'per').replace(',','_').replace(';', '_')

        medname_original_new[med] = new_name
        
    return medname_original_new


def init_med_unit_dict(df, verbose=False):


    med_unit_dict = {}

    for med in df.MedicationID.unique():

        # check if for this med, only one unique unit is used for all patients:
        unique_unit = None
        units_for_this_med = pd.unique(df[df.MedicationID == med].DoseUnitDSC.dropna())
        med_unit_dict[med] = units_for_this_med
        
        if verbose:
            if len(units_for_this_med) == 0:
                print(f'0: {med}')
            elif len(units_for_this_med) > 1:
                print(f'>1: {med}')
                
    return med_unit_dict


def insert_medicationdsc_name_for_template_entries(df):

    med_ids_zzims = df.loc[(df.MedicationDSC == 'ZZ IMS TEMPLATE')].MedicationID.unique()

    for med_id_tmp in med_ids_zzims:
        if med_id_tmp == 800001:
            continue

        names_tmp = list(df.loc[df.MedicationID == med_id_tmp].MedicationDSC.unique())
        names_tmp.remove('ZZ IMS TEMPLATE')
        if len(names_tmp) != 1:

            # see if it's the same type of Med for all, e..g. same med but different doses of med (tablets etc.):
            if not len(np.unique([x.split(' ')[0]] for x in names_tmp)) == 1:
                print(f'Multiple Names for ERX {med_id_tmp}: {names_tmp}')
                continue

        df.loc[(df.MedicationID == med_id_tmp) & 
               (df.MedicationDSC == 'ZZ IMS TEMPLATE'),
               'MedicationDSC'] = names_tmp[0]
        
    return df

In [11]:
def medication_processing_routine(df, remove_original_unit_dose_cols=True, verbose=False):
    '''
    Run the whole medication processing routine. This function contains all the logic how the doses and units are derived from the EDW fields.
    Input:
    df = edw medication data output after preprocess_medication_data() and remove_non_valid_entries()
    Output: 2 dataframes, similar to input but with 3 additional columns: IsInfusion, Dose, Unit
    medications: all entries that could succesfully handeled with the logic implemented
    medications_nonsuccessful: all entries that where a Dose or Unit could not be found (is usually <0.1% of the cases, when clearly sth. is missing in the EDW fields...)
    '''

    df = df.copy()
    df['IsInfusion'] = np.nan
    df['Unit'] = np.nan
    df['Dose'] = np.nan

    print('Manual Correction for phenobarbital ivpb loading dose (100 ml) entries - set dose to mg. Reason: Some of the entries say mg, some mg/kg. So far, all entires should actually be mg. [Done for Covid Sedation Project]:')
    df.loc[(df.MedicationID == 4004082017) & pd.notna(df.SigTXT_Order), 'DoseUnitDSC'] = 'mg'
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)
        print('valid_unit_and_order_dose...')

    print('Manual Correction for Rocuronium entries - set dose to mg. [Done for Covid Delirium Project]:')
    df.loc[(df.MedicationID == 95811) & pd.notna(df.SigTXT_Order) & pd.isna(df.DoseUnitDSC), 'DoseUnitDSC'] = 'mg'
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)
        print('valid_unit_and_order_dose...')
        
    med_unit_dict = init_med_unit_dict(df)
    df = determine_IsInfusion(df)
    df =  valid_unit_and_order_dose(df)
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)
        print('valid_unit_and_discrete_dose...')
    df = valid_unit_and_discrete_dose(df)
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)
        print('icu_sleep_studydrug...')
    df =  icu_sleep_studydrug(df)
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)
        print('valid_infusion_rate_and_dose_from_med_name...')
    df = valid_infusion_rate_and_dose_from_med_name(df)
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)
        ### up to here, usually >95% of the medications can be successfully processed.
        print('self_administered_meds...')
    df = self_administered_meds(df)
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)
        ### up to here, usually > 98% of meds are processed.
        print('patches...')
    df = patches(df)
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)
        print('tablets...')
    df = tablets(df)
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)
        ### up to here, usually > 99% of meds are processed.
        print('impute missing dose units...')
        # impute missing dose units and run above code again
    df = impute_missing_dose_unit(df, med_unit_dict)
    df = valid_unit_and_order_dose(df)
    df = valid_unit_and_discrete_dose(df)
    df = icu_sleep_studydrug(df)
    df = valid_infusion_rate_and_dose_from_med_name(df)
    df = self_administered_meds(df)
    df = patches(df)
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)
        print('use_mean_of_dose_range')
    df = use_mean_of_dose_range(df)
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)
        print('weight dependent doses...')
    df = weight_dependent_dose(df)
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)
        print('sodium chloride ivs...')
                
    df = sodium_chloride_iv(df)
    df = set_isoflurane_to_dummy(df)
    
    if verbose:
        print(df.loc[pd.isna(df.Unit)].shape)

    if remove_original_unit_dose_cols:
        # we have standardized the dose/unit here, so we can remove all EDW dose/unit fields:
        cols_remove = ['DoseUnitCD', 'DoseUnitDSC', 'MinimumDoseAMT', 'MaximumDoseAMT', 'InfusionRateNBR', 'InfusionRateUnitCD', 'InfusionRateUnitDSC', 'DefinedDailyDoseNBR', 'DiscreteDoseAMT', 'DiscreteDispenseUnitDSC']
        cols_remove = [x for x in cols_remove if x in df.columns]
        df.drop(cols_remove, axis = 1, inplace = True)

    df.drop(['DoseUnitDSC_Imputed'], axis = 1, inplace = True)
    
    df['MedicationDSCNew'] = np.nan
    medname_original_new = init_medname_dict(df)
    for original_medname in df.MedicationDSC.unique():
        df.loc[df.MedicationDSC == original_medname, 'MedicationDSCNew'] = medname_original_new[original_medname]

    medications_nonsuccessful = df.loc[pd.isna(df.Unit)]
    medications = df.loc[pd.notna(df.Unit)]

    if verbose:
        print(f'Number of entries that could be successfully processed: {medications.shape[0]}')
        print(f'Number of entries that could not be successfully processed: {medications_nonsuccessful.shape[0]}')
        print(f'All MedicationDSCs that could not be succesfully processed:\n{medications_nonsuccessful.MedicationDSC.value_counts()}')
        
    return medications, medications_nonsuccessful

### only keep medication as defined in medication_dictionary and PatientIDs as defined in cohort


In [12]:
def keep_dictionary_meds_only(meds_data):

    if meds_dictionary_type == 'erx':
        meds_data = meds_data[np.isin(meds_data.MedicationID, list(meds_dictionary['erx']))]

    elif meds_dictionary_type == 'name':

        found_med_bool = meds_data.MedicationDSC.apply(lambda x: any([x_med.lower() in x.lower() for x_med in meds_dictionary['med_name']]))
        meds_data = meds_data.loc[found_med_bool]
        
    return meds_data

print(meds_data.shape)

meds_data = insert_medicationdsc_name_for_template_entries(meds_data)

meds_data = keep_dictionary_meds_only(meds_data)

meds_data = meds_data[np.isin(meds_data.PatientID, cohort.PatientID)]
print(meds_data.shape)

(1483090, 38)
(427988, 38)


In [13]:
# meds_data = meds_data[meds_data.MedicationID == 4004082017]

### basic preprocessing

In [14]:
print(meds_data.shape)
meds_data = preprocess_medication_data(meds_data)    
print(meds_data.shape)
meds_data = remove_non_valid_entries(meds_data, verbose=True)
print(meds_data.shape)

# due to slight changes (e.g. split of composite meds into parts), rerun the 'check if in list':
meds_data = keep_dictionary_meds_only(meds_data)

(427988, 38)
(427988, 32)
Remove entries with "Canceled Entry", "Missed", "Return to Cabinet", or ActionSourceDSC!=MAR
Remove entries related to Anesthesia
Removing the following ear/eye drop meds: 
 [].
Removing the following spray meds: 
 [].
Removing the following millicurie-unit meds: 
 [].
change 'mg of codeine to just mg'
change mg of phosphate to just mg
some manual change of composite drugs is done here:
Some Gel or cream-type of meds are removed due to unclear dose.
Some meds with non-common doses (mEq, Units, puff, mmoll) are removed, not needed currently.
Some more unclear MedicationDSCs are removed.
(410997, 32)


In [15]:
meds_data, meds_nonsuccess = medication_processing_routine(meds_data, remove_original_unit_dose_cols=True, verbose=True)

Manual Correction for phenobarbital ivpb loading dose (100 ml) entries - set dose to mg. Reason: Some of the entries say mg, some mg/kg. So far, all entires should actually be mg. [Done for Covid Sedation Project]:
(410997, 35)
valid_unit_and_order_dose...
Manual Correction for Rocuronium entries - set dose to mg. [Done for Covid Delirium Project]:
(410997, 35)
valid_unit_and_order_dose...
(203995, 35)
valid_unit_and_discrete_dose...
(203469, 35)
icu_sleep_studydrug...
(203469, 35)
valid_infusion_rate_and_dose_from_med_name...
Dose Parsing from MedicationDSC failed for ['isoflurane 99.9 % inhalation liquid', 'voriconazole ivpb in 250 ml'].
(1577, 35)
self_administered_meds...
(1577, 35)
patches...
(1329, 35)
tablets...
(1283, 35)
impute missing dose units...
Dose Parsing from MedicationDSC failed for ['isoflurane 99.9 % inhalation liquid', 'voriconazole ivpb in 250 ml'].
(1013, 36)
use_mean_of_dose_range
(923, 36)
weight dependent doses...
(264, 36)
sodium chloride ivs...
(204, 36)
Ind

In [16]:
def infer_generic_med_name(meds_data):

    meds_data['MedicationGenericName'] = np.nan

    if meds_dictionary_type == 'erx':

        meds_data = meds_data.loc[np.isin(meds_data.MedicationID, meds_dictionary.erx.values)].copy()
        for erx_tmp, name_tmp in zip(meds_dictionary.erx.values, meds_dictionary.med_name.values):
            meds_data.loc[meds_data.MedicationID == erx_tmp, 'MedicationGenericName'] = name_tmp 

    elif meds_dictionary_type == 'name':
        meds_contained = meds_data.MedicationDSC.unique()
        for med_tmp in meds_contained:
            generic_name = [x for x in meds_dictionary.med_name if x.lower() in med_tmp.lower()]
            if (len(generic_name) == 0) & ('_' in med_tmp):
                generic_name = [x for x in meds_dictionary.med_name if x.lower() in med_tmp.replace('_', ' ').lower()]
            if len(generic_name) == 0:
                print(f'No generic medication categories found for {med_tmp}. Skip.')
                continue
            if len(generic_name)>1:    
                # first, remove medication name that are a substring of another name:
                for x in generic_name:
                    if not x in generic_name: 
                        continue #this is werid but has to be done because .remove() is inplace operation.
                    if any([x in y for y in set(generic_name) - set([x])]):
                        generic_name.remove(x)
            if len(generic_name)>1:            
                print(f'Multiple generic medication categories found for {med_tmp}: {generic_name}. Take the first.')
            generic_name = generic_name[0]

            meds_data.loc[meds_data.MedicationDSC == med_tmp,'MedicationGenericName'] = generic_name

    return meds_data


def get_route_from_medname(med_tmp):
    
    if any(['injection' in med_tmp, 'intravenous' in med_tmp, 'infusion' in med_tmp, 
                    'pcea'in med_tmp, 'pca' in med_tmp, 'bag' in med_tmp, 'bolus' in med_tmp, 'ivpb' in med_tmp, 'intramuscular' in med_tmp]):
        cat = 'parenteral'
    elif any(['transdermal' in med_tmp, 'patch' in med_tmp]):
        cat = 'transdermal'
    elif any(['oral' in med_tmp, 'tablet'in med_tmp, 'capsule' in med_tmp, 'inhalation' in med_tmp]):
        cat = 'oral'
    elif any([x in med_tmp for x in ['mg/ml', 'inf']]):
        cat = 'parenteral'
    else:
        cat = None

    return cat


def get_route_from_routedsc(meds_data, med_tmp):
    
    routes = np.unique(meds_data.loc[meds_data.MedicationDSC == med_tmp, 'RouteDSC'])
    if len(routes) == 1:
        route = routes[0]
        if route == 'Intravenous':
            cat = 'parenteral'
        elif route == 'Oral':
            cat = 'oral'
    else:
        print(f'Non unique RouteDSCs for medication {med_tmp}. In previous step, route parsing from name did not succeed either (get_route_from_medname()')
        
    return cat

#### Infer generic medication names and Route
(e.g. 'Propofol', currently only specific Propofol MedicationDSCs are available)

In [17]:
meds_data = infer_generic_med_name(meds_data)

# get Route:
meds_data['Route'] = np.nan
for med_tmp in meds_data['MedicationDSC'].unique():
    route_category = get_route_from_medname(med_tmp)
    if route_category is None:
        route_category = get_route_from_routedsc(meds_data, med_tmp)
    meds_data.loc[meds_data.MedicationDSC==med_tmp, 'Route'] = route_category
    

### Convert Units

In [18]:
def convert_units(dose, unit, body_weight_kg=None):
        # normalize the unit to mg (bolus) or mg/hr (infusion)
    unit = unit.lower()
    
    if unit in ['mg', 'mg/hr']:
        pass
    elif unit in ['g', 'g/hr']:
        dose = dose * 1000.
    elif unit in ['mcg', 'mcg/hr']:
        dose = dose / 1000.
    elif unit == 'mg/min':
        dose = dose * 60.
    elif unit == 'mcg/min':
        dose = dose * 60. / 1000.
    elif unit == 'mcg/hr':
        dose = dose / 1000.
    elif unit == 'mcg/h':
        dose = dose / 1000.
    elif unit == 'mcg/kg/min':
        dose = dose * 60. / 1000.* body_weight_kg
    elif unit == 'mcg/kg/hr':
        dose = dose / 1000.* body_weight_kg
    elif unit == 'mcg/kg':
        dose = dose / 1000.* body_weight_kg
    elif unit == 'mg/kg':
        dose = dose * body_weight_kg
    elif unit == 'g/kg':
        dose = dose * 1000.* body_weight_kg    
    elif unit == 'ml':
        # this is for 'infiltration' and 'nebulization'. does not happen often. this conversino just serves as an estimate (w/o density), results of those meds should be interpreted accordingly.
        dose = dose * 1000
    else:
        raise ValueError('Unknown unit: %s'%(unit))
        
    return dose


def correct_weight_dependent_unit_with_flowsheet_weight(meds, cohort):
    
    # some irregular entries might still have weight_dependent units, fix them with weight obtained from flowsheets:

    df_weight_dependent = meds.loc[meds.Unit.apply(lambda x: '/kg' in x)]
    if df_weight_dependent.shape[0] > 0:
        for jloc, row in df_weight_dependent.iterrows():
            weight = cohort.loc[cohort.PatientID == row.PatientID, 'Weight'].values[0]            
            meds.loc[jloc, 'Dose'] = meds.loc[jloc, 'Dose'] * weight
            meds.loc[jloc, 'Unit'] = meds.loc[jloc, 'Unit'].replace('/kg', '')

    return meds


def convert_to_mg_h(meds):
    
    unit_conversion_to_mg = {
    'mcg': [0.001, 'mg'],
    'mg': [1, 'mg'],
    'g' : [1000, 'mg'],
    
    'mcg/hr': [0.001, 'mg/hr'],
    'mg/hr': [1, 'mg/hr'],
    'g/hr': [1000, 'mg/hr'],
    
    'mcg/min': [0.001 * 60, 'mg/hr'],
    'mg/min': [1 * 60, 'mg/hr'],
    'g/min': [1000 * 60, 'mg/hr']
    }

    for unit in unit_conversion_to_mg:
        meds.loc[meds.Unit == unit, 'Dose'] = meds.loc[meds.Unit == unit, 'Dose'] * unit_conversion_to_mg[unit][0]
        meds.loc[meds.Unit == unit, 'Unit'] = unit_conversion_to_mg[unit][1]
        
    return meds



In [19]:
print(f'Units contained before operation: {pd.unique(meds_data.Unit)}')
meds_data = correct_weight_dependent_unit_with_flowsheet_weight(meds_data, cohort)
meds_data = convert_to_mg_h(meds_data)
print(f'Units contained after operation: {pd.unique(meds_data.Unit)}')

Units contained before operation: ['mg' 'mcg' 'mg/hr' 'mcg/hr' 'mcg/kg/min' 'tablet' 'mg/kg' 'mcg/kg/hr'
 'mcg/kg']
Units contained after operation: ['mg' 'mg/hr' 'tablet']


In [20]:
if 1:
    meds_data.to_csv(os.path.join(meds_output_dir, 'medications_successfully_processed.csv'), index = False)
if 1:
    meds_nonsuccess.to_csv(os.path.join(meds_output_dir, 'medications_not_successfully_processed.csv'), index = False)

In [21]:
if 0:
    meds_data = pd.read_csv(os.path.join(meds_output_dir, 'medications_successfully_processed.csv'))
    for dt_col in ['OrderInstantDTS', 'OrderStartDTS', 'OrderEndDTS', 'MedicationTakenDTS']:
        meds_data[dt_col] = pd.to_datetime(meds_data[dt_col], infer_datetime_format=1)

### Time Series Conversion
We're now able to go from the discrete data view to continuous, time-series like data. this is especially important for infusions.
Equivalency computation happens only afterwards as some medications depend on total daily doses which can be computed from the time series.

In [22]:
def create_timeseries_df_for_subject(df_mrn):
    ''' 
    Input: Medication data for one subject, after running pipeline that computes 'Dose' and 'Unit' columns.
    Ouput: dataframe in 1-minute resolution: Discrete medications in mg, Continuous in mg/min.
    '''

    min_date = np.nanmin([df_mrn.OrderStartDTS.values, df_mrn.MedicationTakenDTS.values])
    min_date = pd.Timestamp(min_date) - timedelta(minutes=1)
    max_date = np.nanmax([df_mrn.OrderEndDTS.values, df_mrn.MedicationTakenDTS.values])

    # initialize new timeseries-based df for this subject:
    meds_ts_df_mrn = pd.DataFrame(index=pd.date_range(min_date, max_date, freq = fs_med)) 

    for med_name in df_mrn.MedicationGenericName:
        meds_ts_df_mrn[med_name] = 0
        meds_ts_df_mrn[med_name + '_oral'] = 0
        meds_ts_df_mrn[med_name + '_parenteral'] = 0
        meds_ts_df_mrn[med_name + '_transdermal'] = 0

    order_ids = pd.unique(df_mrn.OrderID)
    
    for order_id in order_ids:
        df_order = df_mrn[df_mrn.OrderID == order_id].copy()
        meds_ts_df_mrn = timeseries_for_order(meds_ts_df_mrn, df_order)
    
    return meds_ts_df_mrn


def timeseries_for_order(meds_ts_df_mrn, df_order):
    
    order_id = df_order.OrderID.iloc[0]
    med_name_generic = df_order.MedicationGenericName.unique()
    med_name_specific = df_order.MedicationDSC.unique()
    unit = df_order.Unit.dropna().unique()
    route = df_order.Route.dropna().unique()
    assert len(med_name_generic) == 1, f'Different Medication Names within the same OrderID {order_id}'
    assert len(route) == 1, f'Different Routes within the same OrderID {order_id}'
    if len(unit) != 1:
        unit = [df_order.Unit.dropna().value_counts().index[0]]
        print(f'Different Units within the same OrderID {order_id}. Take most occuring: {unit[0]}')
    
    med_name_specific = med_name_specific[0]
    unit = unit[0]
    route = route[0]
    med_name_generic = med_name_generic[0]
    
    type_order = None
    if (unit == 'mg/hr') & ~('patch' in med_name_specific): # (route in ['Intravenous', 'Epidural', 'Intravenous (PCA General)']):
        type_order = 'infusion_iv_continuous'
    elif (unit == 'mg/hr') & ('patch' in med_name_specific):
        type_order = 'patch'
    elif unit == 'mg': # either bolus or tablet
        type_order = 'discrete'
    elif med_name_specific == 'DEXMEDETOMIDINE_OR_PLACEBO_(2017P000090)'.lower():
        type_order = 'icu_sleep_studydrug'
    else:
        print(f'Undefined Type of Med: {df_order.iloc[0]}')


    if type_order == 'infusion_iv_continuous':
        
        tmp = pd.DataFrame(index = meds_ts_df_mrn.index, columns = ['tmp'])
        tmp.iloc[0] = 0
        # infusions are saved in mg/min, so that it matches the timeseries with 1 min sample rate.
        if len(df_order[df_order['Dose']>0]) == 0:
                return meds_ts_df_mrn
        start_time_order = df_order[df_order['Dose']>0].iloc[0].MedicationTakenDTS
        end_time_order = df_order.iloc[-1].OrderEndDTS
        # make sure there's no other entry yet in this timerange for this med:
#         if np.sum(meds_ts_df_mrn.loc[start_time_order : end_time_order, med_name].dropna()) > 0:
#             print(f'Overlapping Order Start and End for: MRN={mrn}, Medication Name={med}, start_time_order={start_time_order}, end_time_order={end_time_order}')
        #  Correct end time order if last entry is set to 0:
        if df_order.iloc[-1].Dose == 0:
            end_time_order = df_order.iloc[-1].MedicationTakenDTS
        df_order = df_order[(df_order.MedicationTakenDTS >= start_time_order) & (df_order.MedicationTakenDTS <= end_time_order)]
        if len(df_order[df_order['Dose']>0]) == 0:
                return meds_ts_df_mrn
        for jloc, row in df_order.iterrows():
            tmp.loc[row.MedicationTakenDTS, 'tmp'] = row.Dose / 60 # mg/min
        # time series with forward-propagate/fill NA values for the continuous meds:
        tmp.loc[end_time_order, 'tmp'] = 0
        tmp['tmp'].fillna(method='ffill', inplace=True)   
        # add it to 'general' med_name:
        meds_ts_df_mrn[med_name_generic] += tmp['tmp'].values
        meds_ts_df_mrn[med_name_generic + '_' + route] += tmp['tmp'].values
    
    elif type_order == 'patch':

        if not 'Patch Applied' in df_order.MARActionDSC.values:
            print('no patch applied?')
            return meds_ts_df_mrn
        
        tmp = pd.DataFrame(index = meds_ts_df_mrn.index, columns = ['tmp'])
        tmp.iloc[0] = 0

        start_time_order = df_order[df_order['MARActionDSC'] == 'Patch Applied'].iloc[0].MedicationTakenDTS
        end_time_order = df_order.iloc[-1].OrderEndDTS
        if df_order.iloc[-1].MARActionDSC == 'Patch Removed':
            end_time_order = df_order.iloc[-1].MedicationTakenDTS

        for jloc, row in df_order.iterrows():
            if row.MARActionDSC in ['Patch Removed', 'Due']:
                assert row.Dose == 0
            tmp.loc[row.MedicationTakenDTS, 'tmp'] = row.Dose / 60

        # if last entry has dose>0, there is no end indicated. use maximum duration derived from DiscreteFrequency. 
        # I think this is necessary because some Orders for patches are for a month, without any entry after a few days any more.
        if row.Dose > 0:
            try:
                duration = np.float(re.search('\d+', row.DiscreteFrequencyDSC)[0])
                unit_duration = [x for x in ['minute', 'hour', 'day'] if x in row.DiscreteFrequencyDSC.lower()][0]
            except:
                print(f'Patch {row.DiscreteFrequencyDSC} max duration parsing failed.')
                return meds_ts_df_mrn
                
            if unit_duration == 'minute':
                duration = duration/60
            elif unit_duration == 'hour':
                duration = duration
            elif unit_duration == 'day':
                duration = duration*24
            else:
                print(f'Unexpected duration unit for Patch {unit_duration}')

            end_dt = row.MedicationTakenDTS + timedelta(hours=duration)
            end_dt = np.min([meds_ts_df_mrn.index[-1], end_dt])
            tmp.loc[end_dt, 'tmp'] = 0
        
        tmp['tmp'].fillna(method='ffill', inplace=True)   
        # add it to 'general' med_name:
        meds_ts_df_mrn[med_name_generic] += tmp['tmp'].values
        meds_ts_df_mrn[med_name_generic + '_' + route] += tmp['tmp'].values

            
    elif type_order == 'discrete':

        for jloc, row in df_order.iterrows():   
            meds_ts_df_mrn.loc[row.MedicationTakenDTS, med_name_generic] += row.Dose
            meds_ts_df_mrn.loc[row.MedicationTakenDTS, med_name_generic + '_' + route] += row.Dose

    elif type_order == 'icu_sleep_studydrug':
        
        tmp = pd.DataFrame(index = meds_ts_df_mrn.index, columns = ['tmp'])
        tmp.iloc[0] = 0

        for jloc, row in df_order.iterrows():
            if row.Dose > 0:
                if row.DurationUnitDSC == 'Minutes':
                    duration_minutes = np.float(row.DurationNBR)
                elif row.DurationUnitDSC == 'Hours':
                    duration_minutes = np.float(row.DurationNBR) * 60
                else:
                    print(f'Unexpected DurationUnitDSC for ICU-Sleep study drug: {row.DurationUnitDSC}')

                end_dt = row.MedicationTakenDTS + timedelta(minutes=duration_minutes)
                tmp.loc[row.MedicationTakenDTS, 'tmp'] = row.Dose / 60

                if pd.isna(meds_ts_df_mrn.loc[end_dt, med_name_generic]):
                    tmp.loc[end_dt, 'tmp'] = 0 # set definitive end point.

            else:
                tmp.loc[row.MedicationTakenDTS, 'tmp'] = 0
        tmp['tmp'].fillna(method='ffill', inplace=True)   
        # add it to 'general' med_name:
        meds_ts_df_mrn[med_name_generic] += tmp['tmp'].values
        meds_ts_df_mrn[med_name_generic + '_' + route] += tmp['tmp'].values

    return meds_ts_df_mrn

In [23]:
meds_data.head()

,OrderID,PatientID,MRN,PatientEncounterID,MedicationID,MedicationDSC,OrderInstantDTS,OrderStartDTS,OrderEndDTS,MedicationTakenDTS,MARActionCD,MARActionDSC,RouteDSC,RouteCD,DurationNBR,DurationUnitCD,DurationUnitDSC,ActionSourceDSC,SigTXT_Order,SigTXT_Administered,MedicationDiscontinueReasonDSC,DiscreteFrequencyDSC,IsInfusion,Unit,Dose,MedicationDSCNew,MedicationGenericName,Route
130,106391599.0,Z13887246,5551589,3.078082e+09,18307.0,gabapentin 400 mg capsule,2015-05-28 22:52:00.0000000,2015-05-28 23:45:00,2015-06-04 19:01:00,2015-05-30 08:18:00,1.0,Given,Oral,15.0,NaN,NaN,NaN,MAR,800.0,NaN,Patient Discharge,Every 8 hours scheduled,False,mg,800.0,gabapentin_400_mg_capsule,gabapentin,oral
53,106391594.0,Z13887246,5551589,3.078082e+09,9637.0,clonazepam 0.5 mg tablet,2015-05-28 22:52:00.0000000,2015-05-28 23:45:00,2015-06-04 19:01:00,2015-05-30 08:20:00,1.0,Given,Oral,15.0,NaN,NaN,NaN,MAR,0.5,NaN,Patient Discharge,2 times daily,False,mg,0.5,clonazepam_0.5_mg_tablet,clonazepam,oral
102,106391604.0,Z13887246,5551589,3.078082e+09,10814.0,oxycodone 5 mg tablet,2015-05-28 22:52:00.0000000,2015-05-28 23:45:00,2015-06-04 19:01:00,2015-05-30 08:20:00,1.0,Given,Oral,15.0,NaN,NaN,NaN,MAR,10.0,NaN,Patient Discharge,2 times daily,False,mg,10.0,oxycodone_5_mg_tablet,oxycodone,oral
131,106391599.0,Z13887246,5551589,3.078082e+09,18307.0,gabapentin 400 mg capsule,2015-05-28 22:52:00.0000000,2015-05-28 23:45:00,2015-06-04 19:01:00,2015-05-30 14:33:00,1.0,Given,Oral,15.0,NaN,NaN,NaN,MAR,800.0,NaN,Patient Discharge,Every 8 hours scheduled,False,mg,800.0,gabapentin_400_mg_capsule,gabapentin,oral
64,106391605.0,Z13887246,5551589,3.078082e+09,10814.0,oxycodone 5 mg tablet,2015-05-28 22:52:00.0000000,2015-05-28 22:46:00,2015-06-04 19:01:00,2015-05-30 18:13:00,1.0,Given,Oral,15.0,NaN,NaN,NaN,MAR,5.0,NaN,Patient Discharge,Every 4 hours PRN,False,mg,5.0,oxycodone_5_mg_tablet,oxycodone,oral


In [24]:
if 0:
    
    equivalency_table = pd.read_csv(equivalency_table_path)

    meds_in_equivalency = [x for x in equivalency_table.med]
    meds_in_equivalency = [re.search(".+_[a-z]+", x)[0] for x in meds_in_equivalency]

    meds_in_data = []
    for med_tmp in meds_data.MedicationGenericName.unique():
        print(med_tmp)
        routes = meds_data.loc[meds_data.MedicationGenericName == med_tmp, 'Route'].unique()
        meds_in_data = meds_in_data + [med_tmp + '_' + x for x in routes]

    meds_in_data_not_equivalency = set(meds_in_data) - set(meds_in_equivalency)
    meds_in_equivalency_not_in_data = set(meds_in_equivalency) - set(meds_in_data)

    meds_in_data_not_equivalency = list(meds_in_data_not_equivalency)
    meds_in_data_not_equivalency.sort()
    print(meds_in_data_not_equivalency)

    meds_in_equivalency_not_in_data = list(meds_in_equivalency_not_in_data)
    meds_in_equivalency_not_in_data.sort()
    print(meds_in_equivalency_not_in_data)


In [25]:
def compute_equivalencies(meds_subject_ts):

    # convert to equivalencies. for each med, get the equivalent factor and multiply the original dose with it.
    # add all routes to generic_equivalency column. if TDD medication, do the procedure per day.

    generic_cols_available = [x for x in meds_subject_ts.columns if all(['_parenteral' not in x,
                                                                        '_oral' not in x,
                                                                        'transdermal' not in x])]

    for category_tmp in equivalency_table.category: # overall categories, like opioids, benzos
        meds_subject_ts[category_tmp] = 0

    for generic_col_tmp in generic_cols_available:
#         print(generic_col_tmp)
#         import pdb; pdb.set_trace()
        generic_equi_col_tmp = generic_col_tmp + '_equi'
        meds_subject_ts[generic_equi_col_tmp] = 0
        calendar_days_contained = pd.Series(meds_subject_ts.iloc[:,0].asfreq('d').index).apply(lambda x: x.date()).unique()

        for route_tmp in ['parenteral', 'oral', 'transdermal']:
            med_tmp = generic_col_tmp + '_' + route_tmp
            if med_tmp in equivalency_table['med'].values:

                equi_table_tmp = equivalency_table[equivalency_table.med == generic_col_tmp + '_' + route_tmp]
                if equi_table_tmp.shape[0] == 1:
                    assert(equi_table_tmp['tdd_dependent'].values[0] == 0)
                    tdd_dependent = False

                    equivalent_factor = equi_table_tmp['equivalency_factor'].values[0]
                    if pd.isna(equivalent_factor): continue
                    meds_subject_ts[generic_equi_col_tmp] += \
                    meds_subject_ts[med_tmp] * equivalent_factor

                    category_tmp = equi_table_tmp['category'].values[0]
                    meds_subject_ts[category_tmp] += \
                    meds_subject_ts[med_tmp] * equivalent_factor               

                if equi_table_tmp.shape[0] > 1:
                    assert( all(equi_table_tmp['tdd_dependent'].values == 1))
                    tdd_dependent = True
                    category_tmp = equi_table_tmp['category'].values[0]

                    med_values_tmp_days = meds_subject_ts[med_tmp].resample('1d').sum()
                    # go through each calendar day :
                    for date_tmp, date_dose_tmp in zip(med_values_tmp_days.index, med_values_tmp_days.values):
                        equi_table_tmp_tdd = equi_table_tmp.loc[(equi_table_tmp.tdd_min <= date_dose_tmp) &
                                                                (date_dose_tmp < equi_table_tmp.tdd_max)]
                        assert(equi_table_tmp_tdd.shape[0] == 1)

                        equivalent_factor = equi_table_tmp_tdd['equivalency_factor'].values[0]
                        if pd.isna(equivalent_factor): continue
                        # generic_med equi:
                        meds_subject_ts.loc[date_tmp.date(): datetime.combine(date_tmp.date(), time(23, 59, 59)), category_tmp] += \
                        meds_subject_ts.loc[date_tmp.date(): datetime.combine(date_tmp.date(), time(23, 59, 59)), med_tmp] * equivalent_factor
                        # and overall category:
                        meds_subject_ts.loc[date_tmp.date(): datetime.combine(date_tmp.date(), time(23, 59, 59)), category_tmp] += \
                        meds_subject_ts.loc[date_tmp.date(): datetime.combine(date_tmp.date(), time(23, 59, 59)), med_tmp] * equivalent_factor
                        
                    
    return meds_subject_ts

In [26]:
# this is done for covid sedation project. might be useful for other projects too though. we are interested in cyp3a4 inhibitors. we don't need exact one, just grouped information if given or not.
# therefore, add a summary column cyp3a4inhibitors that is binary in the timeseries data

cyp3a4inhibitors = ['Aprepitant', 'Atazanavir', 'Cobicistat', 'Indinavir', 'Letermovir', 'Lopinovir_Ritonavir', 'Nelfinavir', 'Ritonavir', 'Ciprofloxacin',
'Clarithromycin', 'Erythromycin', 'Fluconazole', 'Itraconazole', 'Posaconazole', 'Voriconazole', 'Cyclosporine', 'Diltiazem', 'Verapamil', 'Dronedarone', 'Fluvoxamine', 'Nefazodone', 'Imatinib']
cyp3a4inhibitors = [x.lower() for x in cyp3a4inhibitors]

In [27]:
save = True
overwrite = True
fs_med =  '1min'

equivalency_table = pd.read_csv(equivalency_table_path)

meds_ts_df_mrn_all = []

for jloc_subject, row_subject in tqdm(cohort.iterrows()): # .iloc[:5]

    try:
        patient_id = row_subject.PatientID
        meds_subject = meds_data[meds_data.PatientID == patient_id]

        # only data between Admission and Discharge
        meds_subject = meds_subject[meds_subject.MedicationTakenDTS >= row_subject.AdmissionDTS]
        meds_subject = meds_subject[meds_subject.MedicationTakenDTS <= row_subject.DischargeDTS]

        if meds_subject.shape[0] == 0:
            continue

        meds_subject_ts = create_timeseries_df_for_subject(meds_subject)
        meds_subject_ts = compute_equivalencies(meds_subject_ts)

        meds_subject_ts['cyp3a4inhibitor'] = 0
        cyp3a4inhibitors_contained = [x for x in cyp3a4inhibitors if x in meds_subject_ts.columns]
        if len(cyp3a4inhibitors_contained) > 0:
            meds_subject_ts['cyp3a4inhibitor'] = meds_subject_ts[cyp3a4inhibitors_contained].any(axis=1).astype(int)

        meds_subject_ts.to_csv(os.path.join(meds_output_dir_minute_wroute, patient_id + '.csv'), index_label='datetime')
        non_route_cols = [x for x in meds_subject_ts.columns if all(['_parenteral' not in x,
                                                                        '_oral' not in x,
                                                                        '_transdermal' not in x])]
        meds_subject_ts = meds_subject_ts[non_route_cols]
        meds_subject_ts.to_csv(os.path.join(meds_output_dir_minute, patient_id + '.csv'), index_label='datetime')

        # hourly and daily summary:
        meds_subject_ts_hour = meds_subject_ts.resample('1h', label='left').sum()
        meds_subject_ts_day = meds_subject_ts.resample('1d', label='left').sum()

        meds_subject_ts_hour.to_csv(os.path.join(meds_output_dir_hour, patient_id + '.csv'), index_label='datetime')
        meds_subject_ts_day.to_csv(os.path.join(meds_output_dir_day, patient_id + '.csv'), index_label='datetime')
        
    except Exception as e:
        print(f'patient_id {patient_id}')
        print(e)
        continue

154it [11:04,  7.71s/it]

Patch Weekly max duration parsing failed.


338it [20:43,  3.68s/it]


In [28]:
meds_data[np.isin(meds_data.MedicationID, [1004088002, 700612, 16168, 700640, 701565, 6004200032, 40042092, 6004080552, 6004080553])]

,OrderID,PatientID,MRN,PatientEncounterID,MedicationID,MedicationDSC,OrderInstantDTS,OrderStartDTS,OrderEndDTS,MedicationTakenDTS,MARActionCD,MARActionDSC,RouteDSC,RouteCD,DurationNBR,DurationUnitCD,DurationUnitDSC,ActionSourceDSC,SigTXT_Order,SigTXT_Administered,MedicationDiscontinueReasonDSC,DiscreteFrequencyDSC,IsInfusion,Unit,Dose,MedicationDSCNew,MedicationGenericName,Route


In [29]:
meds_output_dir_hour

'/scr/wolfgang/Covid19_Sedation/medications_all_timeseries_hour'